# Сборка llama.cpp с патчем (многоархитектурная) — Google Colab

Собирает **последний семантический релиз** llama.cpp (`vX.Y.Z`) с применением **Pull Request** и публикует результат на **Google Drive**:

```
/MyDrive/llama.cpp_<vX.Y.Z>_pr<NNN>/<arch>/            # распакованная папка
/MyDrive/llama.cpp_<vX.Y.Z>_pr<NNN>/<arch>.tar.gz      # архив (опция)
/MyDrive/llama.cpp_<vX.Y.Z>_pr<NNN>/manifest.json      # манифест (SHA-256 всех файлов)
/MyDrive/llama.cpp_<vX.Y.Z>_pr<NNN>/README.txt
```

## Архитектуры
- `cpu` — OpenBLAS
- `t4` — sm_75
- `l4` — sm_89
- `a100` — sm_80
- `gpu_all` — 75;80;89 (одна fat-сборка)

## Что улучшено относительно исходного скрипта
1. **Инкрементальная сборка** — одно build-дерево (`/content/build`) переиспользуется между архитектурами. При смене `GGML_CUDA_ARCHITECTURES` ninja пересобирает только CUDA-часть → время сокращается в 2–4 раза.
2. **Потоковый вывод в логи** — команды пишут вывод в файл (нет зависаний на буфере pipe, `capture_output=True` удалён).
3. **Атомарная публикация** — сборка идёт локально в `/content/out`, на Drive копируется во временную папку и заменяет старые результаты ТОЛЬКО после успешной сборки и smoke-теста.
4. **Без 0-байтных симлинков** — бинарники и библиотеки копируются с разыменованием ссылок (`cp -L`), поэтому на Drive не лежат пустые `.so`-ссылки.
5. **Манифест** — релиз, коммит, метод наложения PR, флаги, даты, размеры и SHA-256 всех артефактов.
6. **Smoke-тесты** — `ldd` (отсутствующие зависимости) и `--version` для каждого бинарника после каждой сборки.
7. **Ретраи сети и проверка свободного места** — надёжнее на нестабильном соединении Colab.

## Как запускать
1. Откройте в **Google Colab**: меню → Upload notebook (или `File → Upload notebook`).
2. Runtime → **Change runtime type** → **T4 GPU**.
3. **Runtime → Run all**.
4. Подтвердите монтирование Google Drive.
5. Дождитесь «✅ Все операции завершены»; перед закрытием сессии проверьте, что Drive досинхронизировался.

Прогоняйте ячейки **строго по порядку** (или Run all): ячейки образуют единый скрипт.

## 1. Настройки

Все параметры — в одной ячейке: номер PR, список целей, список архитектур, пути, флаги.

## 2. Утилиты

Вспомогательные функции: безопасный запуск команд без deadlock (лог в файл, tail при ошибке), ретраи сети, форматирование размера/времени, SHA-256, проверка дисков, карта флагов архитектур.

## 3. Зависимости и подготовка исходников

- установка системных пакетов (build-essential, cmake, ninja, git, curl, libopenblas);
- клонирование репозитория и checkout **последнего семантического релиза** (`vX.Y.Z`);
- наложение PR по цепочке: `git apply` → `git apply --3way` → cherry-pick коммитов PR → резервное слияние.

## 4. Сборка

Одно build-дерево переиспользуется для всех архитектур: `cmake -G Ninja -B /content/build -S ...` вызывается повторно с новым `GGML_CUDA_ARCHITECTURES`, пересобирается только CUDA-часть. После каждой сборки — smoke-тест.

## 5. Публикация на Google Drive

Локальные артефакты `/content/out/<arch>` копируются во временную папку на Drive (`.new`), затем атомарно заменяют старое содержимое. Дополнительно создаётся архив `<arch>.tar.gz` и манифест.

## 6. Запуск

Запускает весь конвейер. После завершения на Drive будет лежать готовая сборка и манифест.

In [ ]:
# ============================================================
# 1. НАСТРОЙКИ
# ============================================================
import os, shutil, subprocess, sys, time, json, hashlib, tarfile
import platform, requests
from datetime import datetime, timezone

PR_NUMBER = "27537"                 # Pull Request, накладываемый на релиз
REPO_URL = "https://github.com/ggml-org/llama.cpp.git"
REPO_NAME = "llama.cpp"
REPO_OWNER = "ggml-org"

BUILD_TARGETS = [
    "llama-cli",
    "llama-server",
    "llama-bench",
    "llama-perplexity",
    "llama-tokenize",
]

# CUDA-архитектуры
CUDA_ARCH_T4 = "75"
CUDA_ARCH_A100 = "80"
CUDA_ARCH_L4 = "89"
CUDA_ARCH_ALL = "75;80;89"

# Список архитектур (порядок важен: cpu — базовое build-дерево, дальше GPU)
ARCHS = ["cpu", "t4", "l4", "a100", "gpu_all"]

REPO_DIR = "/content/llama.cpp"
DRIVE_ROOT = "/content/drive/MyDrive"
OUT_ROOT = "/content/out"            # локальная площадка для артефактов
LOG_DIR = "/content/logs"
BUILD_DIR = "/content/build"         # одно переиспользуемое build-дерево

FRESH_CLONE = True                   # пересоздавать клон каждый запуск
ALLOW_MERGE_FALLBACK = True          # разрешить резервное слияние ветки PR
TARBALL = True                       # класть на Drive архив <arch>.tar.gz
SMOKE_TEST = True                    # ldd + --version после каждой сборки
JOBS = os.cpu_count() or 2

In [ ]:
# ============================================================
# 2. УТИЛИТЫ
# ============================================================
STAGE_TIMINGS = []


def format_duration(seconds):
    seconds = int(round(seconds))
    if seconds < 60:
        return "%d сек" % seconds
    minutes, sec = divmod(seconds, 60)
    if minutes < 60:
        return "%d мин %d сек" % (minutes, sec)
    hours, minutes = divmod(minutes, 60)
    return "%d ч %d мин" % (hours, minutes)


def human_size(size_bytes):
    size = float(size_bytes)
    for unit in ["Б", "КБ", "МБ", "ГБ", "ТБ"]:
        if size < 1024.0:
            return "%.2f %s" % (size, unit)
        size /= 1024.0
    return "%.2f ПБ" % size


def run(cmd, cwd=None, check=True, quiet=False, label="step"):
    """Потоковый запуск команды без deadlock: вывод пишется в лог-файл."""
    os.makedirs(LOG_DIR, exist_ok=True)
    log_path = os.path.join(LOG_DIR, label + ".log")
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    with open(log_path, "w", encoding="utf-8", errors="replace") as f:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=f,
                             stderr=subprocess.STDOUT, env=env)
        p.wait()
    tail_text = ""
    try:
        with open(log_path, encoding="utf-8", errors="replace") as f:
            tail_text = "".join(f.readlines()[-80:])
    except OSError:
        pass
    if p.returncode != 0:
        print("❌ [%s] код %d:\n%s\n--- tail (%s) ---\n%s"
              % (label, p.returncode, cmd, log_path, tail_text))
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    elif not quiet and tail_text.strip():
        print(tail_text)
    return p


def run_out(cmd, cwd=None, label="out"):
    """Запуск короткой команды и возврат stdout."""
    run(cmd, cwd=cwd, check=False, quiet=True, label=label)
    log_path = os.path.join(LOG_DIR, label + ".log")
    try:
        with open(log_path, encoding="utf-8", errors="replace") as f:
            return f.read()
    except OSError:
        return ""


def retry(cmd, cwd=None, attempts=3, delay=5, label="retry"):
    """Запуск команды с экспоненциальными повторами при сетевых сбоях."""
    for i in range(1, attempts + 1):
        try:
            run(cmd, cwd=cwd, check=True, quiet=True, label=label)
            return
        except subprocess.CalledProcessError as e:
            if i == attempts:
                raise RuntimeError("%s: все %d попыток провалились" % (label, attempts)) from e
            wait = delay * (2 ** (i - 1))
            print("⚠ [%s] попытка %d провалилась; повтор через %d c ..." % (label, i, wait))
            time.sleep(wait)


def timed_step(name, func):
    print("\n▶ Начало: %s" % name)
    start = time.perf_counter()
    res = func()
    elapsed = time.perf_counter() - start
    print("✅ Завершено: %s. Время: %s" % (name, format_duration(elapsed)))
    STAGE_TIMINGS.append((name, elapsed))
    return res


def get_dir_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for fn in files:
            fp = os.path.join(root, fn)
            try:
                if os.path.isfile(fp) or os.path.islink(fp):
                    total += os.path.getsize(fp)
            except OSError:
                pass
    return total


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    try:
        with open(path, "rb") as f:
            while True:
                b = f.read(chunk)
                if not b:
                    break
                h.update(b)
        return h.hexdigest()
    except OSError:
        return ""


def check_disks(min_free_gb=3.0):
    for path, name in [("/content", "локальный диск"), ("/content/drive", "Google Drive")]:
        try:
            st = shutil.disk_usage(path)
            print("Свободно %s (%s): %s" % (name, path, human_size(st.free)))
            if st.free < min_free_gb * 1024 ** 3:
                print("⚠ Мало места на %s: %s" % (name, human_size(st.free)))
        except OSError as e:
            print("⚠ Не удалось проверить %s: %s" % (path, e))


def arch_flags(name):
    def cuda_flags(arch):
        return '-DGGML_CUDA=ON -DGGML_CUDA_ARCHITECTURES="%s"' % arch
    return {
        "cpu": "-DGGML_OPENBLAS=ON",
        "t4": cuda_flags(CUDA_ARCH_T4),
        "l4": cuda_flags(CUDA_ARCH_L4),
        "a100": cuda_flags(CUDA_ARCH_A100),
        "gpu_all": cuda_flags(CUDA_ARCH_ALL),
    }[name]

In [ ]:
# ============================================================
# 3. ЗАВИСИМОСТИ И ПОДГОТОВКА ИСХОДНИКОВ
# ============================================================
def install_dependencies():
    retry("apt-get update -qq", attempts=2, delay=5, label="apt_update")
    run("DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends "
        "build-essential cmake ninja-build git curl libopenblas-dev pkg-config",
        quiet=False, label="apt_install")


def commit_if_changes(message):
    run("git add -A", cwd=REPO_DIR, label="git_add")
    p = run("git diff --cached --quiet", cwd=REPO_DIR, check=False, label="git_diff")
    if p.returncode != 0:
        run("git commit -m '%s'" % message, cwd=REPO_DIR, label="git_commit")
        return True
    return False


def apply_pr(release_tag):
    """Накладывает PR на релиз: diff -> 3way -> cherry-pick -> merge."""
    diff_path = "/tmp/pr.diff"
    diff_url = "https://github.com/%s/%s/pull/%s.diff" % (REPO_OWNER, REPO_NAME, PR_NUMBER)
    print("⬇️ Загрузка diff для PR #%s" % PR_NUMBER)
    retry("curl -fsSL %s -o %s" % (diff_url, diff_path), attempts=3, label="curl_diff")

    p = run("git apply --check --whitespace=nowarn /tmp/pr.diff",
            cwd=REPO_DIR, check=False, label="apply_check")
    if p.returncode == 0:
        run("git apply --whitespace=nowarn /tmp/pr.diff", cwd=REPO_DIR, label="apply")
        if commit_if_changes("Apply PR %s on %s" % (PR_NUMBER, release_tag)):
            return "diff_pr%s" % PR_NUMBER
        return "diff_pr%s_already_present" % PR_NUMBER

    p = run("git apply --check --3way --whitespace=nowarn /tmp/pr.diff",
            cwd=REPO_DIR, check=False, label="apply3_check")
    if p.returncode == 0:
        run("git apply --3way --whitespace=nowarn /tmp/pr.diff",
            cwd=REPO_DIR, label="apply3")
        if commit_if_changes("Apply PR %s 3way on %s" % (PR_NUMBER, release_tag)):
            return "diff3way_pr%s" % PR_NUMBER

    print("Cherry-pick коммитов PR #%s" % PR_NUMBER)
    retry("git fetch origin pull/%s/head" % PR_NUMBER, cwd=REPO_DIR, attempts=3, label="fetch_pr")
    try:
        api_url = ("https://api.github.com/repos/%s/%s/pulls/%s/commits?per_page=100"
                   % (REPO_OWNER, REPO_NAME, PR_NUMBER))
        resp = requests.get(api_url, timeout=60)
        resp.raise_for_status()
        shas = [c.get("sha", "") for c in resp.json() if c.get("sha")]
        run("git cherry-pick --abort", cwd=REPO_DIR, check=False, label="cherry_abort")
        run("git reset --hard", cwd=REPO_DIR, check=False, label="cherry_reset")
        for sha in shas:
            p = run("git cherry-pick -n %s" % sha, cwd=REPO_DIR, check=False, label="cherry_pick")
            if p.returncode != 0:
                run("git cherry-pick --abort", cwd=REPO_DIR, check=False, label="cherry_abort2")
                raise RuntimeError("Не удалось применить коммит %s" % sha[:12])
        if commit_if_changes("Cherry-pick PR %s on %s" % (PR_NUMBER, release_tag)):
            return "cherry_pick_pr%s" % PR_NUMBER
    except Exception as e:
        print("⚠ Cherry-pick не выполнен: %s" % e)
        run("git cherry-pick --abort", cwd=REPO_DIR, check=False, label="cherry_abort3")
        run("git reset --hard", cwd=REPO_DIR, check=False, label="cherry_reset2")

    if ALLOW_MERGE_FALLBACK:
        print("⚠ Резервное слияние ветки PR")
        p = run("git merge FETCH_HEAD --no-edit", cwd=REPO_DIR, check=False, label="merge_pr")
        if p.returncode == 0:
            return "merge_pr%s" % PR_NUMBER
        run("git merge --abort", cwd=REPO_DIR, check=False, label="merge_abort")

    raise RuntimeError("Не удалось применить PR к релизу.")


def prepare_source():
    if FRESH_CLONE and os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR, ignore_errors=True)

    if not os.path.exists(REPO_DIR):
        print("Клонирование %s" % REPO_URL)
        retry("git clone %s %s" % (REPO_URL, REPO_DIR), attempts=3, label="git_clone")

    run("git config --local user.name 'Colab Builder'", cwd=REPO_DIR, label="git_user")
    run("git config --local user.email 'colab.builder@localhost'", cwd=REPO_DIR, label="git_email")

    run("git merge --abort", cwd=REPO_DIR, check=False, label="purge_merge")
    run("git cherry-pick --abort", cwd=REPO_DIR, check=False, label="purge_cherry")

    retry("git fetch --all --tags --prune", cwd=REPO_DIR, attempts=3, label="fetch_tags")
    run("git reset --hard", cwd=REPO_DIR, label="reset_hard")
    run("git clean -fd", cwd=REPO_DIR, label="clean_fd")

    release_tag = run_out("git tag -l 'v*' --sort=-v:refname | head -n 1",
                          cwd=REPO_DIR, label="latest_tag").strip()
    if not release_tag:
        raise RuntimeError("Не удалось определить последний семантический релиз (нет тегов v*).")

    print("Последний релиз: %s" % release_tag)
    run("git checkout %s" % release_tag, cwd=REPO_DIR, quiet=False, label="checkout_tag")

    apply_method = apply_pr(release_tag)
    commit_sha = run_out("git rev-parse --short HEAD", cwd=REPO_DIR, label="commit_sha").strip()
    return release_tag, commit_sha, apply_method

In [ ]:
# ============================================================
# 4. ИНКРЕМЕНТАЛЬНАЯ СБОРКА (одно build-дерево)
# ============================================================
def cmake_configure(flags):
    build_opts = (
        "-DLLAMA_BUILD_TOOLS=ON -DLLAMA_BUILD_TESTS=OFF "
        "-DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=ON "
        "-DLLAMA_BUILD_APP=OFF -DGGML_NATIVE=OFF"
    )
    cmd = ("cmake -G Ninja -B %s -S %s %s "
           "-DCMAKE_BUILD_TYPE=Release -DBUILD_SHARED_LIBS=ON "
           "-DCMAKE_BUILD_RPATH='$ORIGIN' -DCMAKE_INSTALL_RPATH='$ORIGIN' "
           "-DCMAKE_BUILD_WITH_INSTALL_RPATH=ON %s"
           % (BUILD_DIR, REPO_DIR, flags, build_opts))
    run(cmd, quiet=True, label="cmake")


def collect_artifacts(out_dir):
    os.makedirs(out_dir, exist_ok=True)
    # -L: разыменовываем симлинки, чтобы на Drive не было 0-байтных ссылок
    run("cp -rL %s/bin/* %s/" % (BUILD_DIR, out_dir), check=False, label="cp_bin")
    run(r"find %s \( -type f -o -type l \) "
        r"\( -name '*.so*' -o -name 'libllama*' -o -name 'libggml*' \) "
        r"-exec cp -Lf {} %s/ \;" % (BUILD_DIR, out_dir), check=False, label="cp_libs")
    run("chmod -R +x %s" % out_dir, check=False, label="chmod_out")


def smoke_test(out_dir, arch_name):
    if not SMOKE_TEST:
        return True
    ok = True
    for exe in BUILD_TARGETS:
        path = os.path.join(out_dir, exe)
        if not os.path.exists(path):
            print("⚠ [%s] отсутствует %s" % (arch_name, exe))
            continue
        ldd = run_out("ldd %s 2>&1 || true" % path, label="ldd_%s_%s" % (arch_name, exe))
        missing = [ln for ln in ldd.splitlines() if "not found" in ln]
        if missing:
            ok = False
            print("⚠ [%s] %s: отсутствующие зависимости: %s" % (arch_name, exe, missing))
        ver = run_out("%s --version 2>&1 || true" % path,
                      label="ver_%s_%s" % (arch_name, exe)).strip().splitlines()
        first = ver[0] if ver else "?"
        print("   [%s] %s: %s" % (arch_name, exe, first[:100]))
    return ok


def build_arch(arch_name):
    flags = arch_flags(arch_name)
    stage_dir = os.path.join(OUT_ROOT, arch_name)
    if os.path.exists(stage_dir):
        shutil.rmtree(stage_dir, ignore_errors=True)

    print("\n" + "=" * 24 + " Сборка: %s " % arch_name.upper() + "=" * 24)
    cmake_configure(flags)
    start = time.perf_counter()
    run("cmake --build %s --target %s --parallel %d"
        % (BUILD_DIR, " ".join(BUILD_TARGETS), JOBS),
        quiet=True, label="build_%s" % arch_name)
    elapsed = time.perf_counter() - start

    collect_artifacts(stage_dir)
    smoke_test(stage_dir, arch_name)
    size = get_dir_size(stage_dir)
    print("✅ %s: %s | Размер: %s" % (arch_name, format_duration(elapsed), human_size(size)))
    if not os.listdir(stage_dir):
        raise RuntimeError("Сборка %s не произвела артефактов." % arch_name)
    return size, stage_dir, elapsed

In [ ]:
# ============================================================
# 5. ПУБЛИКАЦИЯ НА GOOGLE DRIVE + МАНИФЕСТ
# ============================================================
def make_tarball(stage_dir):
    tar_path = stage_dir + ".tar.gz"
    with tarfile.open(tar_path, "w:gz") as tf:
        tf.add(stage_dir, arcname=os.path.basename(stage_dir))
    return tar_path


def publish_arch(arch_name, stage_dir, output_root):
    """Атомарная публикация: сначала во временную папку `.new`, затем замена."""
    drive_arch = os.path.join(output_root, arch_name)
    tmp_dir = os.path.join(output_root, "." + arch_name + ".new")
    tmp_tar = os.path.join(output_root, "." + arch_name + ".tar.gz.new")
    shutil.rmtree(tmp_dir, ignore_errors=True)
    if os.path.exists(tmp_tar):
        os.remove(tmp_tar)

    print("📤 Копирование %s на Google Drive ..." % arch_name)
    shutil.copytree(stage_dir, tmp_dir)          # разыменование симлинков
    tar_path = None
    if TARBALL:
        tar_path = make_tarball(stage_dir)
        shutil.copy2(tar_path, tmp_tar)

    # старое удаляем ТОЛЬКО после успешной копии нового
    if os.path.exists(drive_arch):
        shutil.rmtree(drive_arch, ignore_errors=True)
    shutil.move(tmp_dir, drive_arch)
    if TARBALL:
        final_tar = os.path.join(output_root, arch_name + ".tar.gz")
        if os.path.exists(final_tar):
            os.remove(final_tar)
        shutil.move(tmp_tar, final_tar)
    print("✅ Опубликовано: %s" % drive_arch)
    return tar_path


def _manifest(artifacts, release_tag, commit_sha, apply_method, start_iso, nvcc_version):
    return {
        "проект": REPO_NAME,
        "источник": REPO_URL,
        "релиз": release_tag,
        "pr": PR_NUMBER,
        "метод_наложения_pr": apply_method,
        "коммит": commit_sha,
        "цели": BUILD_TARGETS,
        "cuda_архитектуры": {
            "t4": CUDA_ARCH_T4, "l4": CUDA_ARCH_L4,
            "a100": CUDA_ARCH_A100, "gpu_all": CUDA_ARCH_ALL,
        },
        "начато": start_iso,
        "завершено": datetime.now(timezone.utc).isoformat(),
        "платформа": platform.platform(),
        "python": platform.python_version(),
        "nvcc": nvcc_version,
        "артефакты": artifacts,
    }


def write_manifest(output_root, artifacts, release_tag, commit_sha, apply_method,
                   start_iso, nvcc_version):
    data = _manifest(artifacts, release_tag, commit_sha, apply_method, start_iso, nvcc_version)
    with open(os.path.join(output_root, "manifest.json"), "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    lines = []
    lines.append("llama.cpp — сборка с PR #%s" % PR_NUMBER)
    lines.append("=" * 44)
    lines.append("Релиз:      %s" % release_tag)
    lines.append("Коммит:     %s" % commit_sha)
    lines.append("Метод PR:   %s" % apply_method)
    lines.append("Начато:     %s" % start_iso)
    lines.append("Завершено:  %s" % datetime.now(timezone.utc).isoformat())
    lines.append("nvcc:       %s" % (nvcc_version or "—"))
    lines.append("")
    lines.append("Сборки (SHA-256 деталей в manifest.json):")
    for name in artifacts:
        a = artifacts[name]
        lines.append("  %-8s %10s  %s" % (name, a.get("размер_МБ", "?"), a.get("лейбл", "")))
    with open(os.path.join(output_root, "README.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print("📄 Манифест записан: %s" % output_root)

In [ ]:
# ============================================================
# 6. ГЛАВНЫЙ СКРИПТ
# ============================================================
def main():
    from google.colab import drive

    total_start = time.perf_counter()
    start_iso = datetime.now(timezone.utc).isoformat()

    print("=" * 78)
    print("СБОРКА llama.cpp: СЕМАНТИЧЕСКИЙ РЕЛИЗ + PR #%s" % PR_NUMBER)
    print("=" * 78)
    print("Цели:         %s" % ", ".join(BUILD_TARGETS))
    print("Архитектуры:  %s" % ", ".join(ARCHS))

    timed_step("Монтирование Google Drive", lambda: drive.mount("/content/drive"))
    timed_step("Проверка свободного места", check_disks)
    timed_step("Установка системных зависимостей", install_dependencies)

    release_tag, commit_sha, apply_method = timed_step(
        "Подготовка исходников (семантический релиз + PR)", prepare_source)

    safe_release = release_tag.replace("/", "-")
    output_root = os.path.join(DRIVE_ROOT, "%s_%s_pr%s" % (REPO_NAME, safe_release, PR_NUMBER))
    os.makedirs(output_root, exist_ok=True)

    print("\nРелиз: %s\nPR: #%s\nМетод: %s\nКоммит: %s\nКаталог: %s"
          % (release_tag, PR_NUMBER, apply_method, commit_sha, output_root))

    cuda_bin = "/usr/local/cuda/bin"
    if os.path.isdir(cuda_bin) and cuda_bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = cuda_bin + os.pathsep + os.environ["PATH"]
    nvcc_path = run_out("which nvcc || true", label="which_nvcc").strip()
    cuda_available = bool(nvcc_path)
    nvcc_version = run_out("nvcc --version 2>&1 || true", label="nvcc_version").strip().splitlines()
    nvcc_version = nvcc_version[-1].strip() if nvcc_version else ""
    if cuda_available:
        print("Найден CUDA: %s (%s)" % (nvcc_path, nvcc_version))
    else:
        print("⚠ CUDA не найден. Будет собрана только CPU-версия.")

    archs = [a for a in ARCHS if a == "cpu" or cuda_available]

    artifacts = {}
    for name in archs:
        size, stage_dir, elapsed = build_arch(name)
        item = {"размер_МБ": round(size / 1024.0 ** 2, 2),
                "время_сборки": format_duration(elapsed),
                "лейбл": stage_dir}
        tar_path = publish_arch(name, stage_dir, output_root)
        if TARBALL and tar_path:
            item["tar_gz"] = "%s.tar.gz" % name
            item["sha256_tar"] = sha256_file(tar_path)
        for exe in BUILD_TARGETS:
            p = os.path.join(stage_dir, exe)
            if os.path.isfile(p):
                item.setdefault("binaries", {})[exe] = sha256_file(p)
        artifacts[name] = item

    write_manifest(output_root, artifacts, release_tag, commit_sha, apply_method,
                   start_iso, nvcc_version)

    total_elapsed = time.perf_counter() - total_start
    print("\n" + "=" * 78)
    print("ИТОГОВЫЙ ОТЧЁТ")
    print("=" * 78)
    print("Релиз: %s | PR: #%s (%s) | Коммит: %s"
          % (release_tag, PR_NUMBER, apply_method, commit_sha))
    print("Каталог: %s" % output_root)
    print("Общее время: %s" % format_duration(total_elapsed))
    print("\nАрхитектура   Размер    Время")
    print("-" * 40)
    for name in archs:
        a = artifacts[name]
        print("%-12s %8s  %9s" % (name, a.get("размер_МБ", "?"), a.get("время_сборки", "")))
    print("\n⏱️ Этапы:")
    for stage, el in STAGE_TIMINGS:
        print("  %-48s %s" % (stage, format_duration(el)))
    print("\n✅ Все операции завершены. Файлы: %s" % output_root)
    print("\n⚠ Google Drive синхронизируется в фоне. Проверьте, что все файлы "
          "загрузились, прежде чем закрывать сессию.")


main()